# SynPUF 2.3M EDA: Complete Analysis

Comprehensive analysis of ALL SynPUF files with df.head(), complete coverage, and visualizations.

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import lzo
import io
import struct
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('bmh')
data_path = r'D:\ADE DATASET DOWNLOAD\SynPUF2.3M'
eda_path = r'D:\ADE DATASET DOWNLOAD\EDA'

## 1. LZO Data Loader

In [ ]:
def load_synpuf_lzo(filename, max_blocks=100):
    path = os.path.join(data_path, filename)
    if not os.path.exists(path): 
        print(f"File not found: {filename}")
        return pd.DataFrame()
        
    with open(path, 'rb') as f:
        content = f.read(1000)
        idx = content.find(b'\x00\x04\x00\x00')
        if idx == -1: 
            idx = content.find(b'\x00\x01\x00\x00')
            if idx == -1: return pd.DataFrame()
            
        f.seek(idx)
        all_data = b''
        for _ in range(max_blocks):
            u_data = f.read(4)
            if not u_data: break
            u_size = struct.unpack('>I', u_data)[0]
            if u_size == 0 or u_size > 1000000: break
            
            c_data = f.read(4)
            if not c_data: break
            c_size = struct.unpack('>I', c_data)[0]
            
            f.read(4)
            comp_data = f.read(c_size)
            if len(comp_data) < c_size: break
            
            if c_size < u_size:
                try:
                    decompressed = lzo.decompress(comp_data, False, u_size)
                    all_data += decompressed
                except: continue
            else:
                all_data += comp_data
        
        try:
            return pd.read_csv(io.BytesIO(all_data), sep=',', low_memory=False)
        except:
            return pd.DataFrame()

## 2. PERSON Table

In [ ]:
person = load_synpuf_lzo('person.5.2.csv.lzo', max_blocks=150)print(f"Total Persons: {len(person):,}")display(person.head(5))if not person.empty and 'gender_concept_id' in person.columns:    person['gender'] = person['gender_concept_id'].map({8507: 'Male', 8532: 'Female'}).fillna('Other')    person['race'] = person['race_concept_id'].map({8516: 'Black', 8527: 'White', 8557: 'Other'}).fillna('Unknown')    person['age'] = 2026 - person['year_of_birth']        fig, axes = plt.subplots(2, 3, figsize=(20, 12))        # Age distribution with median    sns.histplot(person['age'], bins=25, kde=True, ax=axes[0,0], color='skyblue')    axes[0,0].set_title('Age Distribution', fontsize=14, fontweight='bold')    axes[0,0].axvline(person['age'].median(), color='red', linestyle='--', label=f'Median: {person["age"].median():.0f}')    axes[0,0].legend()        # Gender pie chart    gender_counts = person['gender'].value_counts()    axes[0,1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%', startangle=90)    axes[0,1].set_title('Gender Distribution', fontsize=14, fontweight='bold')        # Race bar chart    race_counts = person['race'].value_counts()    sns.barplot(x=race_counts.values, y=race_counts.index, ax=axes[0,2], palette='viridis')    axes[0,2].set_title('Race Distribution', fontsize=14, fontweight='bold')    axes[0,2].set_xlabel('Count')        # Age pyramid by gender    age_bins = pd.cut(person['age'], bins=range(0, 101, 10))    age_gender = person.groupby([age_bins, 'gender']).size().unstack(fill_value=0)    if 'Male' in age_gender.columns and 'Female' in age_gender.columns:        y_pos = np.arange(len(age_gender.index))        axes[1,0].barh(y_pos, -age_gender['Male'], label='Male', color='steelblue')        axes[1,0].barh(y_pos, age_gender['Female'], label='Female', color='coral')        axes[1,0].set_yticks(y_pos)        axes[1,0].set_yticklabels(age_gender.index.astype(str))        axes[1,0].set_xlabel('Population')        axes[1,0].set_ylabel('Age Group')        axes[1,0].set_title('Age Pyramid by Gender', fontsize=14, fontweight='bold')        axes[1,0].legend()        axes[1,0].axvline(0, color='black', linewidth=0.8)        # Age by race boxplot    sns.boxplot(data=person, x='race', y='age', ax=axes[1,1], palette='Set2')    axes[1,1].set_title('Age Distribution by Race', fontsize=14, fontweight='bold')    axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=45)        # Summary table    summary_stats = person.groupby('gender').agg({'age': ['mean', 'median', 'std'], 'person_id': 'count'}).round(2)    axes[1,2].axis('off')    table = axes[1,2].table(cellText=summary_stats.values, rowLabels=summary_stats.index,                           colLabels=['Mean Age', 'Median Age', 'Std Age', 'Count'],                           cellLoc='center', loc='center')    table.auto_set_font_size(False)    table.set_fontsize(10)    table.scale(1, 2)    axes[1,2].set_title('Demographics Summary', fontsize=14, fontweight='bold')        plt.tight_layout()    plt.show()        print(f"\nAge Statistics:")    print(f"  Mean: {person['age'].mean():.1f} years")    print(f"  Median: {person['age'].median():.1f} years")    print(f"  Range: {person['age'].min():.0f} - {person['age'].max():.0f} years")

## 3. VISIT_OCCURRENCE Table

In [ ]:
visits = load_synpuf_lzo('visit_occurrence.5.2.csv.0.lzo', max_blocks=150)print(f"Total Visits: {len(visits):,}")display(visits.head(5))if not visits.empty and 'visit_concept_id' in visits.columns:    visit_map = {9201: 'Inpatient', 9202: 'Outpatient', 9203: 'Emergency', 0: 'General'}    visits['visit_type'] = visits['visit_concept_id'].map(visit_map).fillna('Other')        if 'visit_start_date' in visits.columns:        visits['visit_start_date'] = pd.to_datetime(visits['visit_start_date'], errors='coerce')        visits['year'] = visits['visit_start_date'].dt.year        visits['month'] = visits['visit_start_date'].dt.month        fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Visit type distribution    visit_counts = visits['visit_type'].value_counts()    sns.barplot(x=visit_counts.values, y=visit_counts.index, ax=axes[0,0], palette='rocket')    axes[0,0].set_title('Visit Type Distribution', fontsize=14, fontweight='bold')    axes[0,0].set_xlabel('Count')        # Visits per patient    visits_per_patient = visits.groupby('person_id').size()    sns.histplot(visits_per_patient, bins=30, kde=True, ax=axes[0,1], color='teal')    axes[0,1].set_title('Visits per Patient Distribution', fontsize=14, fontweight='bold')    axes[0,1].set_xlabel('Number of Visits')    axes[0,1].axvline(visits_per_patient.median(), color='red', linestyle='--', label=f'Median: {visits_per_patient.median():.0f}')    axes[0,1].legend()        # Temporal trends    if 'year' in visits.columns:        yearly_visits = visits.groupby(['year', 'visit_type']).size().unstack(fill_value=0)        yearly_visits.plot(kind='line', ax=axes[1,0], marker='o')        axes[1,0].set_title('Visit Trends Over Time', fontsize=14, fontweight='bold')        axes[1,0].set_xlabel('Year')        axes[1,0].set_ylabel('Number of Visits')        axes[1,0].legend(title='Visit Type')        axes[1,0].grid(True, alpha=0.3)        # Monthly patterns    if 'month' in visits.columns:        monthly_pattern = visits.groupby('month').size()        axes[1,1].bar(monthly_pattern.index, monthly_pattern.values, color='coral')        axes[1,1].set_title('Monthly Visit Patterns', fontsize=14, fontweight='bold')        axes[1,1].set_xlabel('Month')        axes[1,1].set_ylabel('Number of Visits')        axes[1,1].set_xticks(range(1, 13))        axes[1,1].grid(True, alpha=0.3, axis='y')        plt.tight_layout()    plt.show()        print(f"\nVisit Statistics:")    print(f"  Average visits per patient: {visits_per_patient.mean():.2f}")    print(f"  Patients with 1 visit: {(visits_per_patient == 1).sum():,}")    print(f"  Patients with 5+ visits: {(visits_per_patient >= 5).sum():,}")

## 4. DRUG_EXPOSURE Table

In [ ]:
drug_exp = load_synpuf_lzo('drug_exposure.5.2.csv.0.lzo', max_blocks=200)print(f"Total Drug Exposures: {len(drug_exp):,}")display(drug_exp.head(5))if not drug_exp.empty and 'person_id' in drug_exp.columns:    patient_drug_counts = drug_exp.groupby('person_id')['drug_concept_id'].nunique().reset_index(name='unique_drugs')    top_drugs = drug_exp['drug_concept_id'].value_counts().head(20)        fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Drug burden distribution    sns.histplot(patient_drug_counts['unique_drugs'], bins=30, kde=True, ax=axes[0,0], color='purple')    axes[0,0].axvline(5, color='red', linestyle='--', linewidth=2, label='Polypharmacy (5+)')    axes[0,0].axvline(patient_drug_counts['unique_drugs'].median(), color='orange', linestyle='--', label=f'Median: {patient_drug_counts["unique_drugs"].median():.0f}')    axes[0,0].set_title('Patient Drug Burden Distribution', fontsize=14, fontweight='bold')    axes[0,0].set_xlabel('Unique Drugs per Patient')    axes[0,0].legend()        # Top drugs    sns.barplot(x=top_drugs.values, y=top_drugs.index, ax=axes[0,1], palette='magma')    axes[0,1].set_title('Top 20 Most Prescribed Drugs', fontsize=14, fontweight='bold')    axes[0,1].set_xlabel('Prescription Count')    axes[0,1].set_ylabel('Drug Concept ID')        # Polypharmacy categories    drug_categories = pd.cut(patient_drug_counts['unique_drugs'],                              bins=[0, 1, 3, 5, 10, float('inf')],                             labels=['1 drug', '2-3 drugs', '4-5 drugs', '6-10 drugs', '10+ drugs'])    cat_counts = drug_categories.value_counts().sort_index()    axes[1,0].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('Set3'))    axes[1,0].set_title('Polypharmacy Categories', fontsize=14, fontweight='bold')        # Drug exposure frequency per patient    exposures_per_patient = drug_exp.groupby('person_id').size()    sns.boxplot(y=exposures_per_patient, ax=axes[1,1], color='lightblue')    axes[1,1].set_title('Drug Exposure Events per Patient', fontsize=14, fontweight='bold')    axes[1,1].set_ylabel('Number of Exposure Events')        plt.tight_layout()    plt.show()        poly_count = (patient_drug_counts['unique_drugs'] >= 5).sum()    print(f"\nDrug Exposure Statistics:")    print(f"  Patients with Polypharmacy (5+ drugs): {poly_count:,} ({poly_count/len(patient_drug_counts)*100:.1f}%)")    print(f"  Average unique drugs per patient: {patient_drug_counts['unique_drugs'].mean():.2f}")    print(f"  Max drugs for one patient: {patient_drug_counts['unique_drugs'].max()}")    print(f"  Total unique drug concepts: {drug_exp['drug_concept_id'].nunique():,}")

## 5. CONDITION_OCCURRENCE Table

In [ ]:
conditions = load_synpuf_lzo('condition_occurrence.5.2.csv.0.lzo', max_blocks=150)print(f"Total Condition Records: {len(conditions):,}")display(conditions.head(5))if not conditions.empty and 'condition_concept_id' in conditions.columns:    top_conditions = conditions['condition_concept_id'].value_counts().head(20)    conditions_per_patient = conditions.groupby('person_id')['condition_concept_id'].nunique()        fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Top conditions    sns.barplot(x=top_conditions.values, y=top_conditions.index, ax=axes[0,0], palette='viridis')    axes[0,0].set_title('Top 20 Condition Concepts', fontsize=14, fontweight='bold')    axes[0,0].set_xlabel('Number of Occurrences')        # Conditions per patient distribution    sns.histplot(conditions_per_patient, bins=30, kde=True, ax=axes[0,1], color='darkgreen')    axes[0,1].set_title('Unique Conditions per Patient', fontsize=14, fontweight='bold')    axes[0,1].set_xlabel('Number of Conditions')    axes[0,1].axvline(conditions_per_patient.median(), color='red', linestyle='--', label=f'Median: {conditions_per_patient.median():.0f}')    axes[0,1].legend()        # Comorbidity levels    comorbidity_levels = pd.cut(conditions_per_patient,                                 bins=[0, 1, 3, 5, 10, float('inf')],                                labels=['1 condition', '2-3 conditions', '4-5 conditions', '6-10 conditions', '10+ conditions'])    comorbidity_counts = comorbidity_levels.value_counts().sort_index()    sns.barplot(x=comorbidity_counts.index, y=comorbidity_counts.values, ax=axes[1,0], palette='RdYlGn_r')    axes[1,0].set_title('Comorbidity Burden Distribution', fontsize=14, fontweight='bold')    axes[1,0].set_xlabel('Comorbidity Level')    axes[1,0].set_ylabel('Number of Patients')    axes[1,0].set_xticklabels(axes[1,0].get_xticklabels(), rotation=45, ha='right')        # Condition occurrences per patient    occurrences_per_patient = conditions.groupby('person_id').size()    sns.violinplot(y=occurrences_per_patient, ax=axes[1,1], color='salmon')    axes[1,1].set_title('Condition Occurrence Events per Patient', fontsize=14, fontweight='bold')    axes[1,1].set_ylabel('Number of Occurrence Events')        plt.tight_layout()    plt.show()        print(f"\nCondition Statistics:")    print(f"  Total unique conditions: {conditions['condition_concept_id'].nunique():,}")    print(f"  Average conditions per patient: {conditions_per_patient.mean():.2f}")    print(f"  Patients with 5+ conditions: {(conditions_per_patient >= 5).sum():,} ({(conditions_per_patient >= 5).sum()/len(conditions_per_patient)*100:.1f}%)")    print(f"  Max conditions for one patient: {conditions_per_patient.max()}")

## 6. DRUG_ERA Table

In [ ]:
drug_era = load_synpuf_lzo('drug_era.csv.1.lzo', max_blocks=150)
print(f"Total Drug Eras: {len(drug_era):,}")
display(drug_era.head(5))

if not drug_era.empty and 'drug_exposure_count' in drug_era.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(drug_era['drug_exposure_count'], bins=30, kde=True)
    plt.title('Drug Exposure Count per Era')
    plt.tight_layout()
    plt.show()

## 7. CONDITION_ERA Table

In [ ]:
condition_era = load_synpuf_lzo('condition_era.csv.1.lzo', max_blocks=150)
print(f"Total Condition Eras: {len(condition_era):,}")
display(condition_era.head(5))

if not condition_era.empty and 'condition_occurrence_count' in condition_era.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(condition_era['condition_occurrence_count'], bins=30, kde=True)
    plt.title('Condition Occurrence Count per Era')
    plt.tight_layout()
    plt.show()

## 8. MEASUREMENT_OCCURRENCE Table

In [ ]:
measurements = load_synpuf_lzo('measurement_occurrence.5.2.csv.0.lzo', max_blocks=150)print(f"Total Measurements: {len(measurements):,}")display(measurements.head(5))if not measurements.empty and 'measurement_concept_id' in measurements.columns:    fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Top measurements    top_measurements = measurements['measurement_concept_id'].value_counts().head(15)    sns.barplot(x=top_measurements.values, y=top_measurements.index, ax=axes[0,0], palette='coolwarm')    axes[0,0].set_title('Top 15 Measurement Types', fontsize=14, fontweight='bold')    axes[0,0].set_xlabel('Count')        # Measurements per patient    meas_per_patient = measurements.groupby('person_id').size()    sns.histplot(meas_per_patient, bins=30, kde=True, ax=axes[0,1], color='navy')    axes[0,1].set_title('Measurements per Patient', fontsize=14, fontweight='bold')    axes[0,1].set_xlabel('Number of Measurements')    axes[0,1].axvline(meas_per_patient.median(), color='yellow', linestyle='--', label=f'Median: {meas_per_patient.median():.0f}')    axes[0,1].legend()        # Value distribution (if numeric values exist)    if 'value_as_number' in measurements.columns:        measurements['value_as_number'] = pd.to_numeric(measurements['value_as_number'], errors='coerce')        valid_values = measurements['value_as_number'].dropna()        if len(valid_values) > 0:            sns.histplot(valid_values, bins=50, kde=True, ax=axes[1,0], color='indigo')            axes[1,0].set_title('Measurement Value Distribution', fontsize=14, fontweight='bold')            axes[1,0].set_xlabel('Value')            axes[1,0].set_xlim(valid_values.quantile(0.01), valid_values.quantile(0.99))        # Unique measurements per patient    unique_meas_per_patient = measurements.groupby('person_id')['measurement_concept_id'].nunique()    sns.boxplot(y=unique_meas_per_patient, ax=axes[1,1], color='lightcoral')    axes[1,1].set_title('Unique Measurement Types per Patient', fontsize=14, fontweight='bold')    axes[1,1].set_ylabel('Unique Measurement Types')        plt.tight_layout()    plt.show()        print(f"\nMeasurement Statistics:")    print(f"  Total unique measurement types: {measurements['measurement_concept_id'].nunique():,}")    print(f"  Average measurements per patient: {meas_per_patient.mean():.2f}")    print(f"  Patients with measurements: {measurements['person_id'].nunique():,}")

## 9. OBSERVATION Table

In [ ]:
observations = load_synpuf_lzo('observation.5.2.csv.0.lzo', max_blocks=150)print(f"Total Observations: {len(observations):,}")display(observations.head(5))if not observations.empty and 'observation_concept_id' in observations.columns:    fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Top observation types    top_obs = observations['observation_concept_id'].value_counts().head(15)    sns.barplot(x=top_obs.values, y=top_obs.index, ax=axes[0,0], palette='plasma')    axes[0,0].set_title('Top 15 Observation Types', fontsize=14, fontweight='bold')    axes[0,0].set_xlabel('Count')        # Observations per patient    obs_per_patient = observations.groupby('person_id').size()    sns.histplot(obs_per_patient, bins=30, kde=True, ax=axes[0,1], color='darkviolet')    axes[0,1].set_title('Observations per Patient', fontsize=14, fontweight='bold')    axes[0,1].set_xlabel('Number of Observations')    axes[0,1].axvline(obs_per_patient.median(), color='lime', linestyle='--', label=f'Median: {obs_per_patient.median():.0f}')    axes[0,1].legend()        # Unique observation types per patient    unique_obs_per_patient = observations.groupby('person_id')['observation_concept_id'].nunique()    sns.violinplot(y=unique_obs_per_patient, ax=axes[1,0], color='gold')    axes[1,0].set_title('Unique Observation Types per Patient', fontsize=14, fontweight='bold')    axes[1,0].set_ylabel('Unique Types')        # Summary table    obs_summary = pd.DataFrame({        'Total Observations': [len(observations)],        'Unique Types': [observations['observation_concept_id'].nunique()],        'Patients with Obs': [observations['person_id'].nunique()],        'Avg per Patient': [obs_per_patient.mean()]    })    axes[1,1].axis('off')    table = axes[1,1].table(cellText=obs_summary.T.values,                            rowLabels=obs_summary.T.index,                           colLabels=['Value'],                           cellLoc='center', loc='center')    table.auto_set_font_size(False)    table.set_fontsize(11)    table.scale(1, 3)    axes[1,1].set_title('Observation Summary Statistics', fontsize=14, fontweight='bold')        plt.tight_layout()    plt.show()        print(f"\nObservation Statistics:")    print(f"  Total unique observation types: {observations['observation_concept_id'].nunique():,}")    print(f"  Average observations per patient: {obs_per_patient.mean():.2f}")    print(f"  Patients with observations: {observations['person_id'].nunique():,}")

## 10. PROCEDURE_OCCURRENCE Table

In [ ]:
procedures = load_synpuf_lzo('procedure_occurrence.5.2.csv.0.lzo', max_blocks=150)print(f"Total Procedures: {len(procedures):,}")display(procedures.head(5))if not procedures.empty and 'procedure_concept_id' in procedures.columns:    top_proc = procedures['procedure_concept_id'].value_counts().head(15)    proc_per_patient = procedures.groupby('person_id')['procedure_concept_id'].nunique()        fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Top procedures    sns.barplot(x=top_proc.values, y=top_proc.index, ax=axes[0,0], palette='tab10')    axes[0,0].set_title('Top 15 Procedures', fontsize=14, fontweight='bold')    axes[0,0].set_xlabel('Count')        # Procedures per patient    proc_counts = procedures.groupby('person_id').size()    sns.histplot(proc_counts, bins=30, kde=True, ax=axes[0,1], color='brown')    axes[0,1].set_title('Procedures per Patient', fontsize=14, fontweight='bold')    axes[0,1].set_xlabel('Number of Procedures')    axes[0,1].axvline(proc_counts.median(), color='red', linestyle='--', label=f'Median: {proc_counts.median():.0f}')    axes[0,1].legend()        # Unique procedure types per patient    sns.boxplot(y=proc_per_patient, ax=axes[1,0], color='olive')    axes[1,0].set_title('Unique Procedure Types per Patient', fontsize=14, fontweight='bold')    axes[1,0].set_ylabel('Unique Types')        # Procedure distribution pie chart    proc_categories = pd.cut(proc_per_patient,                              bins=[0, 1, 3, 5, 10, float('inf')],                             labels=['1', '2-3', '4-5', '6-10', '10+'])    cat_counts = proc_categories.value_counts().sort_index()    axes[1,1].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%', startangle=90)    axes[1,1].set_title('Procedure Burden Categories', fontsize=14, fontweight='bold')        plt.tight_layout()    plt.show()        print(f"\nProcedure Statistics:")    print(f"  Total unique procedures: {procedures['procedure_concept_id'].nunique():,}")    print(f"  Average procedures per patient: {proc_counts.mean():.2f}")    print(f"  Patients with 5+ procedure types: {(proc_per_patient >= 5).sum():,}")

## 11. Summary Report

In [ ]:
summary = {    'Table': ['PERSON', 'VISIT_OCCURRENCE', 'DRUG_EXPOSURE', 'CONDITION_OCCURRENCE',               'DRUG_ERA', 'CONDITION_ERA', 'MEASUREMENT', 'OBSERVATION', 'PROCEDURE'],    'Records': [len(person), len(visits), len(drug_exp), len(conditions),                len(drug_era), len(condition_era), len(measurements), len(observations), len(procedures)]}summary_df = pd.DataFrame(summary)display(summary_df)fig, axes = plt.subplots(1, 2, figsize=(18, 7))# Dataset coverage bar chartsns.barplot(data=summary_df, x='Records', y='Table', ax=axes[0], palette='mako')axes[0].set_title('SynPUF Dataset Coverage', fontsize=14, fontweight='bold')axes[0].set_xlabel('Number of Records')for i, v in enumerate(summary_df['Records']):    axes[0].text(v, i, f' {v:,}', va='center', fontsize=9)# Proportional representationsummary_df_sorted = summary_df.sort_values('Records', ascending=False)axes[1].pie(summary_df_sorted['Records'], labels=summary_df_sorted['Table'],            autopct=lambda pct: f'{pct:.1f}%' if pct > 2 else '',           startangle=90, colors=sns.color_palette('mako', len(summary_df)))axes[1].set_title('Proportional Data Distribution', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print("\n" + "="*60)print("SYNPUF 2.3M COMPREHENSIVE ANALYSIS COMPLETE")print("="*60)print(f"Total Records Analyzed: {summary_df['Records'].sum():,}")print(f"Total Tables Covered: {len(summary_df)}")print("="*60)